In [ ]:
%%shell
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mv StyleTTS2-LJSpeech/Audio .

In [1]:
import torch
import argparse
from tqdm import tqdm
import cProfile
import os

# Import your new specialized modules
from Adversarial_TTS.model_loader import initialize_environment, load_optimizer
from Adversarial_TTS.core_logic import run_optimization_generation
from Adversarial_TTS.reporting import finalize_run

class NotebookArgs:
    def __init__(self):
        # String parameters
        self.ground_truth_text = "I think the NFL is lame and boring"
        self.target_text = "The Seattle Seahawks are the best Team in the world"

        # Numeric parameters
        self.loop_count = 1
        self.num_generations = 4
        self.pop_size = 4
        self.iv_scalar = 0.5
        self.size_per_phoneme = 1
        self.batch_size = -1

        # Boolean parameters
        self.notify = False
        self.subspace_optimization = False

        # Enum/Selection parameters
        self.mode = "TARGETED"
        self.ACTIVE_OBJECTIVES = ["PESQ", "WHISPER_PROB"]
        self.thresholds = ["PESQ=0.2", "WHISPER_PROB=0.6"]

args = NotebookArgs()

In [2]:
# 1. Set Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# 2. Initialize Environment
# This handles Enums, Thresholds, Model Loading, and Reference Data generation
config_data, model_data, audio_data, embedding_data = initialize_environment(args, device)

if config_data is None:
    print("Initialization failed.")
else:
    print(f"Starting Optimization Loop...")

Using device: cpu
=== Configuration ===
Mode:              TARGETED
GT Text:           I think the NFL is lame and boring
Target Text:       The Seattle Seahawks are the best Team in the world
Generations:       4
Population Size:   4
Loop Count:        1
IV Scalar:         0.5
Size Per Phoneme:  1
Notify (WhatsApp): False
Objectives:        ['PESQ', 'WHISPER_PROB']
Thresholds:        PESQ<=0.2, WER_GT<=0.6
Loading StyleTTS2...
Loading ASR Model...
Starting Optimization Loop...


In [3]:
# 4. Main Loop
for iteration in tqdm(range(config_data.loop_count), desc="Total Progress"):
    profiler = cProfile.Profile()
    profiler.enable()

    # Run the generation loop (Core Logic)
    fitness_data, progress_bar, stop_optimization, gen = run_optimization_generation(
        config_data, model_data, audio_data, embedding_data, iteration, device
    )

    # 5. Finalize and Save Results
    # This creates folders, saves audio, generates graphs, and sends notifications
    folder_path = finalize_run(config_data, fitness_data, model_data, audio_data, progress_bar, gen, device)

    model_data.optimizer = load_optimizer(audio_data, config_data)

    profiler.disable()
    profiler.dump_stats(os.path.join(folder_path, "performance_data.prof"))

Current Generation 1: 100%|██████████| 4/4 [08:20<00:00, 121.36s/it]
                                                                    /home/yanis/miniconda3/envs/styletts2/lib/python3.9/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
Total Progress: 100%|██████████| 1/1 [08:27<00:00, 507.90s/it]
